# Detectability Table

This notebook builds a species-level detectability table using:

- IUCN species data (`ab_thres = True` only)
- IUCN camera lookup data  
- SSUSA observations  

The final output includes:

- `sci_name`
- `IUCN_lookup_count`
- `SSUSA_camera_count`

In [79]:
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

In [80]:
# Paths
# -----------------------------
OUTPUT_PATH = "../../outputs"
CLEANED_PATH = "../cleaned"
PREPROCESSED_PATH = "../preprocessed_data"

# -----------------------------
# Parameters
# -----------------------------
BUFFER_KM = 1

# -----------------------------
# Input files
# -----------------------------
SSUSA_CSV = "ssusa_cleaned.csv"
CAMERA_FOOTPRINTS = f"ssusa_camera_footprints_{BUFFER_KM}km.geojson"
IUCN_LOOKUP_CSV = f"iucn_camera_species_lookup_{BUFFER_KM}km.csv"
IUCN_DATA = "iucn_cleaned.shp"

# -----------------------------
# Coordinate columns
# -----------------------------
LON_COL = "Longitude"
LAT_COL = "Latitude"

In [82]:
# Load cleaned SSUSA observations
ssusa = pd.read_csv(os.path.join(CLEANED_PATH, SSUSA_CSV), low_memory=False)

# Load camera footprint polygons
camera_fp = gpd.read_file(os.path.join(OUTPUT_PATH, CAMERA_FOOTPRINTS))

# Load IUCN camera footprint-species lookup table
iucn_lookup = pd.read_csv(os.path.join(PREPROCESSED_PATH, IUCN_LOOKUP_CSV))

# Load cleaned IUCN species data
iucn_data = gpd.read_file(os.path.join(CLEANED_PATH, IUCN_DATA))

In [83]:
print("SSUSA shape:", ssusa.shape)
print("Camera footprints shape:", camera_fp.shape)
print("IUCN lookup shape:", iucn_lookup.shape)
print("IUCN species shape:", iucn_data.shape)

SSUSA shape: (713319, 29)
Camera footprints shape: (7303, 3)
IUCN lookup shape: (321727, 4)
IUCN species shape: (733, 28)


### Keep IUCN Species Where ab_thres == True

In [84]:
# Keep species where ab_thres = True
species_table = (
    iucn_data.loc[iucn_data["ab_thres"] == True, ["sci_name"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

print(" IUCN species body mass > 500 g:", len(species_table))
species_table.head()

 IUCN species body mass > 500 g: 127


,sci_name
0,cuniculus paca
2,alouatta pigra
4,antilocapra americana
9,aplodontia rufa
17,ateles geoffroyi


### Count IUCN - Camera footprint spatial join Lookups per Species

In [85]:
# print unique species in lookup
print("Unique species in IUCN lookup:", iucn_lookup["sci_name"].nunique())


Unique species in IUCN lookup: 303


In [86]:
# Count lookup rows per species
lookup_counts = (
    iucn_lookup[["sci_name"]]
    .dropna()
    .groupby("sci_name")
    .size()
    .reset_index(name="IUCN_polygons_intersecting")
)

print("Species in lookup:", lookup_counts.shape[0])
lookup_counts.head()

Species in lookup: 303


,sci_name,IUCN_polygons_intersecting
0,alces alces,834
1,alexandromys oeconomus,42
2,ammospermophilus harrisii,84
3,ammospermophilus interpres,11
4,ammospermophilus leucurus,150


In [87]:
# Keep only valid IUCN species that also appear in the lookup table
species_table = species_table.merge(
    lookup_counts,
    on="sci_name",
    how="inner"
)

print("Species after lookup filter:", species_table.shape[0])
species_table.head()

Species after lookup filter: 68


,sci_name,IUCN_polygons_intersecting
0,antilocapra americana,630
1,aplodontia rufa,251
2,canis latrans,7101
3,canis lupus,595
4,canis rufus,34


### Count SSUSA Camera Locations

In [88]:
def make_camera_id(lon, lat):
    """Create a unique camera ID from coordinates."""
    return f"{lon:.8f}_{lat:.8f}"

In [89]:
# Create camera IDs for SSUSA observations
ssusa["coord_id"] = ssusa.apply(
    lambda row: make_camera_id(row[LON_COL], row[LAT_COL]),
    axis=1
)

print("Unique SSUSA cameras:", ssusa["coord_id"].nunique())

Unique SSUSA cameras: 7340


### Count unique camera locations per species 

In [90]:
# Count unique camera locations per species
obs_counts = (
    ssusa[["coord_id", "Species_Name"]]
    .dropna()
    .drop_duplicates()
    .groupby("Species_Name")["coord_id"]
    .nunique()
    .reset_index(name="SSUSA_camera_count")
    .rename(columns={"Species_Name": "sci_name"})
)

obs_counts.head()

,sci_name,SSUSA_camera_count
0,alces alces,165
1,ammospermophilus harrisii,17
2,ammospermophilus leucurus,56
3,antilocapra americana,130
4,aplodontia rufa,13


In [91]:
# Merge SSUSA counts into species table
species_table = species_table.merge(
    obs_counts,
    on="sci_name",
    how="left"
)

# Fill missing species with zero observations
species_table["SSUSA_camera_count"] = (
    species_table["SSUSA_camera_count"]
    .fillna(0)
    .astype(int)
)

species_table.head()

,sci_name,IUCN_polygons_intersecting,SSUSA_camera_count
0,antilocapra americana,630,130
1,aplodontia rufa,251,13
2,canis latrans,7101,3136
3,canis lupus,595,110
4,canis rufus,34,5


### Add Common names for the species 


In [92]:
from get_common_names import get_common_names

# Get unique species names from final table
species_list = species_table["sci_name"].dropna().unique().tolist()

# Retrieve common names
common_names = get_common_names(species_list)


In [93]:
# Convert dictionary to dataframe
common_names_df = pd.DataFrame(
    list(common_names.items()),
    columns=["sci_name", "common_name"]
)

# Merge into final table
species_table = species_table.merge(
    common_names_df,
    on="sci_name",
    how="left"
)

species_table.head()

,sci_name,IUCN_polygons_intersecting,SSUSA_camera_count,common_name
0,antilocapra americana,630,130,Pronghorn
1,aplodontia rufa,251,13,Mountain Beaver
2,canis latrans,7101,3136,Coyote
3,canis lupus,595,110,Wolf
4,canis rufus,34,5,Red Wolf


In [95]:
# Rearrange final columns
species_table = species_table[
    [
        "sci_name",
        "common_name",
        "IUCN_polygons_intersecting",
        "SSUSA_camera_count"
    ]
]

species_table.head()

,sci_name,common_name,IUCN_polygons_intersecting,SSUSA_camera_count
0,antilocapra americana,Pronghorn,630,130
1,aplodontia rufa,Mountain Beaver,251,13
2,canis latrans,Coyote,7101,3136
3,canis lupus,Wolf,595,110
4,canis rufus,Red Wolf,34,5


In [96]:
# Save final detectability table
outfile = os.path.join(
    OUTPUT_PATH,
    f"species_detectability_IUCN_SSUSA.csv"
)

species_table.to_csv(outfile, index=False)

print("Saved file:", outfile)
species_table.head()

Saved file: ../../outputs/species_detectability_IUCN_SSUSA.csv


,sci_name,common_name,IUCN_polygons_intersecting,SSUSA_camera_count
0,antilocapra americana,Pronghorn,630,130
1,aplodontia rufa,Mountain Beaver,251,13
2,canis latrans,Coyote,7101,3136
3,canis lupus,Wolf,595,110
4,canis rufus,Red Wolf,34,5
